# Livy REST calls from Python (no sparkmagic)

Fallback and companion to the lab: the REST calls you typed with `curl`, wrapped in a small Python class based on `requests`.
It needs only `pip install requests`. If your Knox certificate comes from an internal CA, set `REQUESTS_CA_BUNDLE`
to the CA file (PEM) **before** starting Jupyter.

In [ ]:
import getpass, json, time
import requests, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

livy_url = input("Livy URL (from your handout, no trailing slash): ").strip().rstrip("/")
ws_user = input("User name (WORKSHOP_USER): ").strip()
ldap_password = getpass.getpass(f"LDAP password for {ws_user}: ")
workshop_root = input("WORKSHOP_ROOT [/2026_cdp_workshop]: ").strip() or "/2026_cdp_workshop"
workshop_home = f"/user/{ws_user}"

## The client

Each method is one REST call from the lab: `create_session` = `POST /sessions`, `wait_for` = polling `/state`,
`run` = `POST /sessions/<id>/statements` plus polling until the statement is `available`, `delete_session` = `DELETE`.
The `X-Requested-By` header is required by Livy for `POST` and `DELETE`.

In [ ]:
class Livy:
    def __init__(self, url, user, password):
        self.url = url.rstrip("/")
        self.http = requests.Session()
        self.http.auth = (user, password)  # HTTP Basic against Knox (LDAP)
        self.http.verify = False  # skip certificate verification (workshop only)
        self.http.headers.update({"Content-Type": "application/json", "X-Requested-By": user})

    def call(self, method, path, body=None):
        r = self.http.request(method, self.url + path, data=json.dumps(body) if body is not None else None)
        if r.status_code == 401:
            raise RuntimeError("401 from Knox: wrong user name or LDAP password")
        if r.status_code == 403:
            raise RuntimeError("403 from Knox: your user may not use the Livy service")
        r.raise_for_status()
        return r.json()

    def create_session(self, name, **options):
        body = {"kind": "pyspark", "name": name, **options}
        return self.call("POST", "/sessions", body)

    def wait_for(self, sid, wanted=("idle",), timeout=300):
        deadline = time.time() + timeout
        while time.time() < deadline:
            state = self.call("GET", f"/sessions/{sid}/state")["state"]
            print("session", sid, "state:", state)
            if state in wanted:
                return state
            if state in ("dead", "error", "killed", "shutting_down"):
                raise RuntimeError(f"session {sid} ended in state {state}; see GET /sessions/{sid}/log")
            time.sleep(5)
        raise TimeoutError("session did not become idle in time")

    def run(self, sid, code, timeout=600):
        stmt = self.call("POST", f"/sessions/{sid}/statements", {"code": code})
        deadline = time.time() + timeout
        while time.time() < deadline:
            stmt = self.call("GET", f"/sessions/{sid}/statements/{stmt['id']}")
            if stmt["state"] in ("available", "error", "cancelled"):
                break
            time.sleep(2)
        out = stmt.get("output") or {}
        if out.get("status") == "ok":
            print(out["data"].get("text/plain", ""))
        else:
            print("STATEMENT ERROR:", out.get("ename"), out.get("evalue"))
        return stmt

    def delete_session(self, sid):
        return self.call("DELETE", f"/sessions/{sid}")


livy = Livy(livy_url, ws_user, ldap_password)
print("sessions on the server:", livy.call("GET", "/sessions")["total"])

## Open a session

The name must be unique on the Livy server. `conf` passes Spark properties; here it enables the Hive Metastore.
If the name exists from an earlier run, delete the old session first (`livy.call("GET", "/sessions")` shows its id).

In [ ]:
session = livy.create_session(
    f"livy-nb-rest-{ws_user}",
    driverMemory="1g", executorMemory="1g", executorCores=1, numExecutors=1,
    conf={"spark.sql.catalogImplementation": "hive"},
)
sid = session["id"]
livy.wait_for(sid)
info = livy.call("GET", f"/sessions/{sid}")
print("YARN application:", info["appId"], "| owner:", info["owner"], "| proxyUser:", info["proxyUser"])

## Run the pipeline

The code is an ordinary Python string. The driver does not see this notebook's variables, so the paths are put into the text
(f-string) before it is sent — the same idea as the real paths you wrote into the request files in the lab. The code below contains no curly braces of its own.

In [ ]:
livy.run(sid, f"""
from pyspark.sql import functions as F
meter = (spark.read.option("header", True)
    .schema(
        "ts_utc STRING, station_id STRING, device_id STRING, phase STRING, "
        "active_power_kw DOUBLE, reactive_power_kvar DOUBLE, voltage_v DOUBLE, "
        "current_a DOUBLE, power_factor DOUBLE, frequency_hz DOUBLE, "
        "transformer_load_pct DOUBLE, quality_flag STRING, event_code STRING, "
        "comm_status STRING")
    .csv("{workshop_root}/data/grid_measurements/sample-grid-measurements.csv")
    .withColumn("ts_utc", F.to_timestamp(F.col("ts_utc"))))
weather = (spark.read.option("header", True)
    .schema(
        "ts_utc STRING, weather_station_id STRING, station_name STRING, "
        "temperature_c DOUBLE, relative_humidity_pct INT, wind_speed_mps DOUBLE, "
        "precipitation_mm DOUBLE, pressure_hpa DOUBLE, weather_quality STRING")
    .csv("{workshop_root}/data/operator_weather/sample-operator-weather.csv")
    .withColumn("ts_utc", F.to_timestamp(F.col("ts_utc"))))
print("meter rows:", meter.count(), "| weather rows:", weather.count())
""")

In [ ]:
livy.run(sid, f"""
meter_ok = (meter.filter(F.col("quality_flag") == "OK")
    .dropDuplicates(["ts_utc", "station_id", "device_id", "phase"]))
curated = meter_ok.join(weather, ["ts_utc"], "inner")
print("cleansed:", meter_ok.count(), "| curated:", curated.count())
curated.write.mode("overwrite").partitionBy("station_id").parquet("{workshop_home}/livy-nb-rest-curated")
""")

## A statement that fails

The REST call succeeds; the error is part of the statement result, and the session stays usable.

In [ ]:
livy.run(sid, "meter.select(\"no_such_column\").show()")
print("session state:", livy.call("GET", f"/sessions/{sid}/state")["state"])

## Close the session

A session you forget keeps its YARN containers until Livy's idle timeout. Always delete it.

In [ ]:
print(livy.delete_session(sid))

## Optional: a batch

The script from the lab lives on HDFS. A batch needs no session: you give the file and arguments, then poll the state.
Uncomment and run when `${WORKSHOP_ROOT}/09-livy/assets/spark_cleanse_join.py` exists.

In [ ]:
# batch = livy.call("POST", "/batches", {
#     "file": f"hdfs://{workshop_root}/09-livy/assets/spark_cleanse_join.py",
#     "name": f"livy-nb-batch-{ws_user}",
#     "args": [
#         f"{workshop_root}/data/grid_measurements/sample-grid-measurements.csv",
#         f"{workshop_root}/data/operator_weather/sample-operator-weather.csv",
#         f"{workshop_root}/data/awattar/sample-awattar-prices.csv",
#         f"{workshop_home}/livy-nb-batch-curated",
#     ],
# })
# while True:
#     state = livy.call("GET", f"/batches/{batch['id']}/state")["state"]
#     print("batch", batch["id"], state)
#     if state in ("success", "dead", "killed"):
#         break
#     time.sleep(5)